# Cache Equivalence


> Compare no-cache and cached sweep execution for the current interference MCF scale case.


This notebook checks whether caching through the reminder phase changes the raw recalls or derived summaries for the active interference sweep.


In [1]:
import csv
import os
import warnings
from pathlib import Path

import numpy as np
from IPython.display import Markdown
from jax import random

from jaxcmr.analyses.spc import fixed_pres_spc
from jaxcmr.helpers import find_project_root
from selective_interference_v2 import (
    Paradigm,
    compute_n_presented,
    film_recalled_stats,
    load_fit_params,
    make_factory,
    make_is_emotional,
    prepare_sweep,
    run_sweep,
    split_scales_for_cache,
    standard_remap,
)

warnings.filterwarnings("ignore")


In [2]:
# --- sweep configuration ---
SWEEP_PARAM = "interference_mcf_scale"
SWEEP_VALUES = [0.0, 0.33, 0.67, 1.0, 1.33, 1.67, 2.0, 2.33, 2.67, 3.0]
SWEEP_LABEL_FMT = "{:.2f}"
SWEEP_MOTIVATION = ""

# --- paradigm geometry ---
N_FILM = 16
N_BREAK = 0
N_INTERFERENCE = 16
N_FILLER = 16
N_BREAK_MAX = 32
N_INTERFERENCE_MAX = 32
N_FILLER_MAX = 32
EXPERIMENT_COUNT = 5000
SHOW_BREAK_IN_SPC = False
SHOW_FILLERS_IN_SPC = True

# --- emotional mechanism ---
FILM_EMOTIONAL = False
INTERFERENCE_EMOTIONAL = False

# --- cache comparison ---
REFERENCE_CACHE_AFTER = "creation"
CANDIDATE_CACHE_AFTER = "reminder"

# --- fixed scale factors ---
FIXED_SCALES = {
    "break_drift_scale": 1.0,
    "break_mcf_scale": 1.0,
    "reminder_start_drift_scale": 0.0,
    "reminder_drift_scale": 0.0,
    "filler_drift_scale": 1.0,
    "filler_mcf_scale": 1.0,
}

# --- parameter loading ---
SEED = 0
DATA_TAG = "Dupertuys2026"
MODEL_NAME = "eCMR_source_only_phi"
RUN_TAG = "learn_after_encoding_free_recall_1_diffw_0p5_1p0_rtol_1e-4_pooled_evosax_positional_term_best_of_1"
FIT_DIR = "results/fits"
FIT_PATH = ""

# --- output ---
VALIDATION_DIR = "results/validations"
VALIDATION_STR = "cache_equivalence_interference_mcf_scale"


In [3]:
#| output: asis
if SWEEP_MOTIVATION:
    display(Markdown(SWEEP_MOTIVATION))


In [4]:
sweep_values = np.array(SWEEP_VALUES)

paradigm = Paradigm(
    n_film=N_FILM,
    n_break=N_BREAK,
    n_interference=N_INTERFERENCE,
    n_filler=N_FILLER,
    n_break_max=N_BREAK_MAX,
    n_interference_max=N_INTERFERENCE_MAX,
    n_filler_max=N_FILLER_MAX,
    experiment_count=EXPERIMENT_COUNT,
)

project_root = Path(find_project_root())
if FIT_PATH:
    fit_path = project_root / FIT_PATH
else:
    fit_path = project_root / FIT_DIR / f"{DATA_TAG}_{MODEL_NAME}_{RUN_TAG}.json"

validation_dir = str(project_root / VALIDATION_DIR)
os.makedirs(validation_dir, exist_ok=True)

is_emotional = make_is_emotional(
    paradigm,
    film_emotional=FILM_EMOTIONAL,
    interference_emotional=INTERFERENCE_EMOTIONAL,
)
factory = make_factory(is_emotional=is_emotional)
params, n_subjects = load_fit_params(fit_path)
n_presented = compute_n_presented(
    paradigm,
    show_break=SHOW_BREAK_IN_SPC,
    show_fillers=SHOW_FILLERS_IN_SPC,
)

print(f"{n_subjects} subjects")
print(f"reference cache_after={REFERENCE_CACHE_AFTER}")
print(f"candidate cache_after={CANDIDATE_CACHE_AFTER}")
print(f"fit path: {fit_path}")


1 subjects
reference cache_after=creation
candidate cache_after=reminder
fit path: /Users/jordangunn/workspace/selective_interference_v2/results/fits/Dupertuys2026_eCMR_source_only_phi_learn_after_encoding_free_recall_1_diffw_0p5_1p0_rtol_1e-4_pooled_evosax_positional_term_best_of_1.json


In [5]:
def run_cache(cache_after):
    pre_cache_scales, post_cache_scales = split_scales_for_cache(FIXED_SCALES, cache_after)
    prepared = prepare_sweep(
        params,
        paradigm,
        factory,
        cache_after=cache_after,
        tier="standard",
        **pre_cache_scales,
    )
    recalls_4d, _ = run_sweep(
        prepared,
        random.PRNGKey(SEED),
        **{**post_cache_scales, SWEEP_PARAM: sweep_values},
    )
    print(f"{cache_after} pre-cache scales: {pre_cache_scales}")
    print(f"{cache_after} post-cache scales: {post_cache_scales}")
    return recalls_4d, pre_cache_scales, post_cache_scales


reference_recalls_4d, reference_pre_cache_scales, reference_post_cache_scales = run_cache(
    REFERENCE_CACHE_AFTER
)
candidate_recalls_4d, candidate_pre_cache_scales, candidate_post_cache_scales = run_cache(
    CANDIDATE_CACHE_AFTER
)


creation pre-cache scales: {}
creation post-cache scales: {'break_drift_scale': 1.0, 'break_mcf_scale': 1.0, 'reminder_start_drift_scale': 0.0, 'reminder_drift_scale': 0.0, 'filler_drift_scale': 1.0, 'filler_mcf_scale': 1.0}


reminder pre-cache scales: {'break_drift_scale': 1.0, 'break_mcf_scale': 1.0, 'reminder_start_drift_scale': 0.0, 'reminder_drift_scale': 0.0}
reminder post-cache scales: {'filler_drift_scale': 1.0, 'filler_mcf_scale': 1.0}


In [6]:
def summarize_recalls(recalls_4d):
    spcs = []
    stats = []
    for i in range(recalls_4d.shape[0]):
        recalls = recalls_4d[i].reshape(-1, recalls_4d.shape[-1])
        recalls = standard_remap(recalls, paradigm)
        spcs.append(fixed_pres_spc(recalls, n_presented))
        stats.append(film_recalled_stats(recalls, paradigm, n_subjects))
    return spcs, stats


reference_spcs, reference_stats = summarize_recalls(reference_recalls_4d)
candidate_spcs, candidate_stats = summarize_recalls(candidate_recalls_4d)

reference_array = np.asarray(reference_recalls_4d)
candidate_array = np.asarray(candidate_recalls_4d)
mismatch_mask = reference_array != candidate_array
raw_mismatch_count = int(np.sum(mismatch_mask))
raw_equal = raw_mismatch_count == 0

by_value_rows = []
max_spc_abs_diff = 0.0
max_film_mean_abs_diff = 0.0
for value, ref_spc, cand_spc, ref_stats, cand_stats in zip(
    sweep_values,
    reference_spcs,
    candidate_spcs,
    reference_stats,
    candidate_stats,
):
    spc_abs_diff = float(np.nanmax(np.abs(np.asarray(ref_spc) - np.asarray(cand_spc))))
    film_mean_diff = float(ref_stats[0] - cand_stats[0])
    film_mean_abs_diff = abs(film_mean_diff)
    max_spc_abs_diff = max(max_spc_abs_diff, spc_abs_diff)
    max_film_mean_abs_diff = max(max_film_mean_abs_diff, film_mean_abs_diff)
    by_value_rows.append(
        [
            f"{value:g}",
            ref_stats[0],
            cand_stats[0],
            film_mean_diff,
            film_mean_abs_diff,
            spc_abs_diff,
        ]
    )

passed = raw_equal and max_spc_abs_diff == 0.0 and max_film_mean_abs_diff == 0.0

summary_path = os.path.join(validation_dir, f"{VALIDATION_STR}_summary.csv")
by_value_path = os.path.join(validation_dir, f"{VALIDATION_STR}_by_value.csv")
mismatch_path = os.path.join(validation_dir, f"{VALIDATION_STR}_mismatches.csv")

summary_rows = [
    ["reference_cache_after", REFERENCE_CACHE_AFTER],
    ["candidate_cache_after", CANDIDATE_CACHE_AFTER],
    ["raw_equal", raw_equal],
    ["raw_mismatch_count", raw_mismatch_count],
    ["max_spc_abs_diff", max_spc_abs_diff],
    ["max_film_mean_abs_diff", max_film_mean_abs_diff],
    ["passed", passed],
    ["n_values", len(sweep_values)],
    ["n_subjects", n_subjects],
    ["experiment_count", EXPERIMENT_COUNT],
]

with open(summary_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["metric", "value"])
    writer.writerows(summary_rows)

with open(by_value_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(
        [
            "value",
            "reference_film_mean",
            "candidate_film_mean",
            "film_mean_diff",
            "film_mean_abs_diff",
            "max_spc_abs_diff",
        ]
    )
    writer.writerows(by_value_rows)

if raw_mismatch_count:
    with open(mismatch_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(
            [
                "value_index",
                "subject_index",
                "replication_index",
                "recall_index",
                "reference",
                "candidate",
            ]
        )
        for value_index, subject_index, replication_index, recall_index in np.argwhere(
            mismatch_mask
        ):
            writer.writerow(
                [
                    value_index,
                    subject_index,
                    replication_index,
                    recall_index,
                    reference_array[
                        value_index, subject_index, replication_index, recall_index
                    ],
                    candidate_array[
                        value_index, subject_index, replication_index, recall_index
                    ],
                ]
            )
elif os.path.exists(mismatch_path):
    os.remove(mismatch_path)

print(f"summary: {summary_path}")
print(f"by-value: {by_value_path}")
if raw_mismatch_count:
    print(f"mismatches: {mismatch_path}")
display(Markdown(f"**Cache equivalence:** {'PASS' if passed else 'FAIL'}"))


summary: /Users/jordangunn/workspace/selective_interference_v2/results/validations/cache_equivalence_interference_mcf_scale_summary.csv
by-value: /Users/jordangunn/workspace/selective_interference_v2/results/validations/cache_equivalence_interference_mcf_scale_by_value.csv


**Cache equivalence:** PASS